# Counsly 2026 Prediction Training

This Colab notebook downloads historical TNEA data from GitHub raw CSV URLs, builds prediction features, trains two TensorFlow models, validates on 2025, and exports 2026 prediction CSVs plus model artifacts.

Models included:
- Rank-band classifier: predicts general-rank band and community-rank band from aggregate mark and community.
- College cutoff regressor: predicts closing aggregate mark and closing general rank for each college, branch, reservation category, and round.


In [ ]:
# Colab setup and configuration
!pip install -q pandas numpy scikit-learn tensorflow joblib matplotlib requests

from __future__ import annotations

import json
import math
import os
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import requests
import tensorflow as tf
from google.colab import files
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

# Paste GitHub raw URLs here before running the notebook.
GRL_CSV_URL = "https://raw.githubusercontent.com/MDHaarith/Counsly/main/supabase_db/Data_Extractor/data/merged_general_rank_list_all_years.csv"
ALLOTMENT_CSV_URL = "https://raw.githubusercontent.com/MDHaarith/Counsly/main/supabase_db/Data_Extractor/data/merged_records_all_years_rounds_training_ready.csv"
BOARD_DIFFICULTY_CSV_URL = "https://raw.githubusercontent.com/MDHaarith/Counsly/main/supabase_db/Data_Extractor/Pass_Percentage/result_analysis_docs/hse_plus2_result_analysis_2020_2025_summary.csv"

TARGET_YEAR = 2026
VALIDATION_YEAR = 2025
MARK_GRID_STEP = 0.25
RANDOM_SEED = 42

DATA_DIR = Path("/content/counsly_data")
OUTPUT_DIR = Path("/content/counsly_outputs")
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)
print("TensorFlow", tf.__version__)


In [ ]:
# Download and load source CSV files
def download_file(url: str, output_path: Path) -> Path:
    if not url.strip():
        raise ValueError(f"Missing URL for {output_path.name}. Fill GRL_CSV_URL, ALLOTMENT_CSV_URL, and BOARD_DIFFICULTY_CSV_URL in the config cell.")
    response = requests.get(url, timeout=120)
    response.raise_for_status()
    output_path.write_bytes(response.content)
    return output_path

grl_path = download_file(GRL_CSV_URL, DATA_DIR / "merged_general_rank_list_all_years.csv")
allotment_path = download_file(ALLOTMENT_CSV_URL, DATA_DIR / "merged_records_all_years_rounds_training_ready.csv")
board_difficulty_path = download_file(BOARD_DIFFICULTY_CSV_URL, DATA_DIR / "hse_plus2_result_analysis_2020_2025_summary.csv")

grl_raw = pd.read_csv(grl_path, encoding="utf-8-sig")
allotment_raw = pd.read_csv(allotment_path, encoding="utf-8-sig")
board_difficulty_raw = pd.read_csv(board_difficulty_path, encoding="utf-8-sig")

print("GRL rows:", len(grl_raw), grl_raw.columns.tolist())
print("Allotment rows:", len(allotment_raw), allotment_raw.columns.tolist())
print("Board difficulty rows:", len(board_difficulty_raw), board_difficulty_raw.columns.tolist())


In [ ]:
# Shared helpers
COMMUNITY_MAP = {"MBCDNC": "MBC", "MBCV": "MBC", "SCA": "SC"}
RANK_BAND_NAMES = ["top_1", "top_5", "top_10", "top_25", "top_50", "bottom_50"]

def normalize_community(value: object) -> str:
    text = str(value).strip().upper()
    return COMMUNITY_MAP.get(text, text)

def percentile_to_band(percentile: float) -> str:
    if pd.isna(percentile):
        return "unknown"
    if percentile <= 0.01:
        return "top_1"
    if percentile <= 0.05:
        return "top_5"
    if percentile <= 0.10:
        return "top_10"
    if percentile <= 0.25:
        return "top_25"
    if percentile <= 0.50:
        return "top_50"
    return "bottom_50"

def add_history_features(df: pd.DataFrame, key_cols: list[str], value_cols: list[str]) -> pd.DataFrame:
    out = df.sort_values([*key_cols, "year"]).copy()
    grouped = out.groupby(key_cols, dropna=False, sort=False)
    for col in value_cols:
        out[f"prev_{col}"] = grouped[col].shift(1)
        out[f"roll3_median_{col}"] = grouped[col].transform(lambda s: s.shift(1).rolling(3, min_periods=1).median())
        out[f"roll3_std_{col}"] = grouped[col].transform(lambda s: s.shift(1).rolling(3, min_periods=2).std())
        out[f"trend_{col}"] = grouped[col].transform(lambda s: s.shift(1) - s.shift(2))
    out["history_count"] = grouped.cumcount()
    out["missing_history_count"] = 3 - out["history_count"].clip(upper=3)
    return out

def add_prior_group_stats(df: pd.DataFrame, group_cols: list[str], value_col: str, prefix: str) -> pd.DataFrame:
    out = df.sort_values([*group_cols, "year"]).copy()
    grouped = out.groupby(group_cols, dropna=False, sort=False)[value_col]
    out[f"{prefix}_prev"] = grouped.transform(lambda s: s.shift(1))
    out[f"{prefix}_roll3_median"] = grouped.transform(lambda s: s.shift(1).rolling(3, min_periods=1).median())
    out[f"{prefix}_roll3_std"] = grouped.transform(lambda s: s.shift(1).rolling(3, min_periods=2).std())
    out[f"{prefix}_trend"] = grouped.transform(lambda s: s.shift(1) - s.shift(2))
    return out

def prepare_board_difficulty_features(board_df: pd.DataFrame) -> pd.DataFrame:
    if board_df.empty:
        return pd.DataFrame({"year": [TARGET_YEAR]})
    df = board_df.copy()
    df.columns = [c.strip().lower() for c in df.columns]
    for col in ["year", "appeared_total", "passed_total", "pass_percentage", "boys_pass_percentage", "girls_pass_percentage"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.sort_values("year")
    max_appeared = df["appeared_total"].max()
    df["hse2_appeared_total"] = df["appeared_total"]
    df["hse2_pass_percentage"] = df["pass_percentage"]
    df["hse2_fail_percentage"] = 100.0 - df["pass_percentage"]
    df["hse2_competition_index"] = df["appeared_total"] / max_appeared
    df["hse2_pass_percentage_change"] = df["pass_percentage"].diff().fillna(0.0)
    df["hse2_appeared_change"] = df["appeared_total"].diff().fillna(0.0)
    df["hse2_gender_pass_gap"] = df["girls_pass_percentage"] - df["boys_pass_percentage"]

    if TARGET_YEAR not in set(df["year"].dropna().astype(int)):
        recent = df.tail(3)
        future = {"year": TARGET_YEAR}
        for col in ["hse2_appeared_total", "hse2_pass_percentage", "hse2_fail_percentage", "hse2_competition_index", "hse2_pass_percentage_change", "hse2_appeared_change", "hse2_gender_pass_gap"]:
            future[col] = float(recent[col].median()) if col in recent else 0.0
        df = pd.concat([df, pd.DataFrame([future])], ignore_index=True, sort=False)

    keep = ["year", "hse2_appeared_total", "hse2_pass_percentage", "hse2_fail_percentage", "hse2_competition_index", "hse2_pass_percentage_change", "hse2_appeared_change", "hse2_gender_pass_gap"]
    return df[keep].fillna(0.0)

def to_dense_float32(matrix):
    if hasattr(matrix, "toarray"):
        matrix = matrix.toarray()
    return matrix.astype("float32")

def balanced_sample_weights(y: np.ndarray) -> np.ndarray:
    labels, counts = np.unique(y, return_counts=True)
    weights = {label: len(y) / (len(labels) * count) for label, count in zip(labels, counts)}
    return np.array([weights[label] for label in y], dtype="float32")

def make_single_rank_classifier(input_dim: int, output_size: int, name: str, regularized: bool) -> tf.keras.Model:
    inputs = tf.keras.Input(shape=(input_dim,), name="features")
    if regularized:
        regularizer = tf.keras.regularizers.l2(1e-4)
        x = tf.keras.layers.Dense(256, activation="relu", kernel_regularizer=regularizer, name=f"{name}_dense_256")(inputs)
        x = tf.keras.layers.BatchNormalization(name=f"{name}_bn_256")(x)
        x = tf.keras.layers.Dropout(0.35, name=f"{name}_dropout_256")(x)
        x = tf.keras.layers.Dense(128, activation="relu", kernel_regularizer=regularizer, name=f"{name}_dense_128")(x)
        x = tf.keras.layers.Dropout(0.30, name=f"{name}_dropout_128")(x)
    else:
        x = tf.keras.layers.Dense(128, activation="relu", name=f"{name}_dense_128")(inputs)
        x = tf.keras.layers.Dropout(0.20, name=f"{name}_dropout_128")(x)
        x = tf.keras.layers.Dense(64, activation="relu", name=f"{name}_dense_64")(x)
    outputs = tf.keras.layers.Dense(output_size, activation="softmax", name="rank_band")(x)
    model = tf.keras.Model(inputs=inputs, outputs=outputs, name=f"{name}_rank_classifier")
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

def make_dense_regressor(input_dim: int, output_dim: int) -> tf.keras.Model:
    inputs = tf.keras.Input(shape=(input_dim,), name="features")
    x = tf.keras.layers.Dense(1024, activation="relu", name="dense_1024_in")(inputs)
    x = tf.keras.layers.BatchNormalization(name="bn_1024_in")(x)
    x = tf.keras.layers.Dropout(0.25, name="dropout_1024_in")(x)

    shortcut = x
    x = tf.keras.layers.Dense(1024, activation="relu", name="dense_1024_res_1")(x)
    x = tf.keras.layers.BatchNormalization(name="bn_1024_res_1")(x)
    x = tf.keras.layers.Dropout(0.20, name="dropout_1024_res_1")(x)
    x = tf.keras.layers.Dense(1024, activation="relu", name="dense_1024_res_2")(x)
    x = tf.keras.layers.Add(name="residual_add_1024")([shortcut, x])
    x = tf.keras.layers.BatchNormalization(name="bn_1024_out")(x)

    x = tf.keras.layers.Dense(512, activation="relu", name="dense_512")(x)
    x = tf.keras.layers.Dropout(0.20, name="dropout_512")(x)
    x = tf.keras.layers.Dense(256, activation="relu", name="dense_256")(x)
    x = tf.keras.layers.Dense(128, activation="relu", name="dense_128")(x)
    outputs = tf.keras.layers.Dense(output_dim, name="targets")(x)
    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss="mse", metrics=["mae"])
    return model


In [ ]:
# Build rank-band training data and 2026 input frame
def build_rank_feature_tables(grl: pd.DataFrame, board_difficulty: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    df = grl.copy()
    df.columns = [c.strip().upper() for c in df.columns]
    df = df.rename(columns={"GENERAL RANK": "general_rank", "AGGREGATE MARK": "aggregate_mark", "COMMUNITY": "community", "COMMUNITY RANK": "community_rank", "YEAR": "year"})
    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
    df["general_rank"] = pd.to_numeric(df["general_rank"], errors="coerce")
    df["community_rank"] = pd.to_numeric(df["community_rank"], errors="coerce")
    df["aggregate_mark"] = pd.to_numeric(df["aggregate_mark"], errors="coerce").round(2)
    df["community"] = df["community"].map(normalize_community)
    df = df.dropna(subset=["year", "general_rank", "community_rank", "aggregate_mark", "community"])
    df["year"] = df["year"].astype(int)

    year_size = df.groupby("year")["general_rank"].max().rename("year_cohort_size")
    community_size = df.groupby(["year", "community"])["community_rank"].max().rename("community_cohort_size")
    df = df.join(year_size, on="year").join(community_size, on=["year", "community"])
    df["general_percentile"] = df["general_rank"] / df["year_cohort_size"]
    df["community_percentile"] = df["community_rank"] / df["community_cohort_size"]

    grouped = (
        df.groupby(["year", "aggregate_mark", "community"], as_index=False)
        .agg(
            best_general_rank=("general_rank", "min"),
            median_general_rank=("general_rank", "median"),
            best_community_rank=("community_rank", "min"),
            median_community_rank=("community_rank", "median"),
            best_general_percentile=("general_percentile", "min"),
            best_community_percentile=("community_percentile", "min"),
            mark_frequency=("general_rank", "size"),
            year_cohort_size=("year_cohort_size", "max"),
            community_cohort_size=("community_cohort_size", "max"),
        )
    )
    grouped["general_rank_band"] = grouped["best_general_percentile"].map(percentile_to_band)
    grouped["community_rank_band"] = grouped["best_community_percentile"].map(percentile_to_band)

    historical_marks = grouped["aggregate_mark"].dropna().unique()
    grid_marks = np.round(np.arange(0.0, 200.0 + MARK_GRID_STEP, MARK_GRID_STEP), 2)
    inference_marks = np.unique(np.concatenate([historical_marks, grid_marks]))
    communities = sorted(grouped["community"].dropna().unique())
    future = pd.MultiIndex.from_product([[TARGET_YEAR], inference_marks, communities], names=["year", "aggregate_mark", "community"]).to_frame(index=False)

    panel = pd.concat([grouped, future], ignore_index=True, sort=False)
    board_features = prepare_board_difficulty_features(board_difficulty)
    panel = panel.merge(board_features, on="year", how="left")
    panel = add_history_features(panel, ["aggregate_mark", "community"], ["best_general_percentile", "best_community_percentile", "year_cohort_size", "community_cohort_size", "mark_frequency"])

    feature_defaults = [c for c in panel.columns if c.startswith(("prev_", "roll3_", "trend_"))]
    for col in feature_defaults:
        panel[col] = panel[col].fillna(0.0)
    panel["missing_history_count"] = panel["missing_history_count"].fillna(3).astype(float)

    training = panel[panel["year"] < TARGET_YEAR].dropna(subset=["general_rank_band", "community_rank_band"]).copy()
    future_inputs = panel[panel["year"] == TARGET_YEAR].copy()
    return training, future_inputs

rank_training, rank_2026_input = build_rank_feature_tables(grl_raw, board_difficulty_raw)
rank_training.to_csv(OUTPUT_DIR / "rank_band_training.csv", index=False)
rank_2026_input.to_csv(OUTPUT_DIR / "rank_band_2026_input.csv", index=False)
print("Rank training rows:", len(rank_training))
print("Rank 2026 input rows:", len(rank_2026_input))
rank_training.head()


In [ ]:
# Train rank-band classifier and export 2026 predictions
rank_numeric_features = [
    "year", "aggregate_mark", "missing_history_count",
    "hse2_appeared_total", "hse2_pass_percentage", "hse2_fail_percentage", "hse2_competition_index", "hse2_pass_percentage_change", "hse2_appeared_change", "hse2_gender_pass_gap",
    "prev_best_general_percentile", "roll3_median_best_general_percentile", "roll3_std_best_general_percentile", "trend_best_general_percentile",
    "prev_best_community_percentile", "roll3_median_best_community_percentile", "roll3_std_best_community_percentile", "trend_best_community_percentile",
    "prev_year_cohort_size", "roll3_median_year_cohort_size", "prev_community_cohort_size", "roll3_median_community_cohort_size",
    "prev_mark_frequency", "roll3_median_mark_frequency",
]
rank_categorical_features = ["community"]

for col in rank_numeric_features:
    if col not in rank_training.columns:
        rank_training[col] = 0.0
        rank_2026_input[col] = 0.0
    rank_training[col] = pd.to_numeric(rank_training[col], errors="coerce").fillna(0.0)
    rank_2026_input[col] = pd.to_numeric(rank_2026_input[col], errors="coerce").fillna(0.0)

rank_preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), rank_numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), rank_categorical_features),
    ]
)

rank_train_df = rank_training[rank_training["year"] < VALIDATION_YEAR].copy()
rank_val_df = rank_training[rank_training["year"] == VALIDATION_YEAR].copy()

X_rank_train = to_dense_float32(rank_preprocessor.fit_transform(rank_train_df[rank_numeric_features + rank_categorical_features]))
X_rank_val = to_dense_float32(rank_preprocessor.transform(rank_val_df[rank_numeric_features + rank_categorical_features]))
X_rank_2026 = to_dense_float32(rank_preprocessor.transform(rank_2026_input[rank_numeric_features + rank_categorical_features]))

general_band_encoder = LabelEncoder().fit(RANK_BAND_NAMES)
community_band_encoder = LabelEncoder().fit(RANK_BAND_NAMES)
y_general_train = general_band_encoder.transform(rank_train_df["general_rank_band"])
y_community_train = community_band_encoder.transform(rank_train_df["community_rank_band"])
y_general_val = general_band_encoder.transform(rank_val_df["general_rank_band"])
y_community_val = community_band_encoder.transform(rank_val_df["community_rank_band"])
community_sample_weight = balanced_sample_weights(y_community_train)

general_rank_model = make_single_rank_classifier(X_rank_train.shape[1], len(RANK_BAND_NAMES), "general", regularized=False)
community_rank_model = make_single_rank_classifier(X_rank_train.shape[1], len(RANK_BAND_NAMES), "community", regularized=True)
general_rank_callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", mode="max", patience=10, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-6),
]
community_rank_callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", mode="max", patience=12, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-6),
]
general_rank_model.fit(
    X_rank_train,
    y_general_train,
    validation_data=(X_rank_val, y_general_val),
    epochs=80,
    batch_size=512,
    callbacks=general_rank_callbacks,
    verbose=1,
)
community_rank_model.fit(
    X_rank_train,
    y_community_train,
    validation_data=(X_rank_val, y_community_val),
    sample_weight=community_sample_weight,
    epochs=150,
    batch_size=512,
    callbacks=community_rank_callbacks,
    verbose=1,
)

general_val_proba = general_rank_model.predict(X_rank_val, verbose=0)
community_val_proba = community_rank_model.predict(X_rank_val, verbose=0)
general_val_pred = general_val_proba.argmax(axis=1)
community_val_pred = community_val_proba.argmax(axis=1)

general_2026_proba = general_rank_model.predict(X_rank_2026, verbose=0)
community_2026_proba = community_rank_model.predict(X_rank_2026, verbose=0)
rank_predictions = rank_2026_input[["year", "aggregate_mark", "community"]].copy()
rank_predictions["predicted_general_rank_band"] = general_band_encoder.inverse_transform(general_2026_proba.argmax(axis=1))
rank_predictions["predicted_general_rank_band_confidence"] = general_2026_proba.max(axis=1)
rank_predictions["predicted_community_rank_band"] = community_band_encoder.inverse_transform(community_2026_proba.argmax(axis=1))
rank_predictions["predicted_community_rank_band_confidence"] = community_2026_proba.max(axis=1)
rank_predictions.to_csv(OUTPUT_DIR / "rank_band_2026_predictions.csv", index=False)

general_rank_model.save(OUTPUT_DIR / "general_rank_band_model.keras")
community_rank_model.save(OUTPUT_DIR / "community_rank_band_model.keras")
joblib.dump({"preprocessor": rank_preprocessor, "general_band_encoder": general_band_encoder, "community_band_encoder": community_band_encoder, "features": rank_numeric_features + rank_categorical_features}, OUTPUT_DIR / "rank_feature_pipeline.joblib")

rank_metrics = {
    "general_band_accuracy": float(accuracy_score(y_general_val, general_val_pred)),
    "general_band_macro_f1": float(f1_score(y_general_val, general_val_pred, average="macro")),
    "community_band_accuracy": float(accuracy_score(y_community_val, community_val_pred)),
    "community_band_macro_f1": float(f1_score(y_community_val, community_val_pred, average="macro")),
    "train_rows": int(len(rank_train_df)),
    "validation_rows": int(len(rank_val_df)),
    "prediction_rows_2026": int(len(rank_predictions)),
}
rank_metrics


In [ ]:
# Build college cutoff training data and 2026 input frame
def build_cutoff_feature_tables(allotment: pd.DataFrame, board_difficulty: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    df = allotment.copy()
    df.columns = [c.strip().upper() for c in df.columns]
    df = df.rename(columns={
        "YEAR": "year",
        "ROUND": "round",
        "AGGREGATE MARK": "aggregate_mark",
        "RANK": "rank",
        "COMMUNITY": "student_community",
        "COLLEGE CODE": "college_code",
        "BRANCH CODE": "branch_code",
        "ALLOTTED CATEGORY": "allotted_category",
    })
    for col in ["year", "round", "aggregate_mark", "rank"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    for col in ["college_code", "branch_code", "allotted_category", "student_community"]:
        df[col] = df[col].astype(str).str.strip().str.upper()
    df["allotted_category"] = df["allotted_category"].map(normalize_community)
    df["student_community"] = df["student_community"].map(normalize_community)
    df = df.dropna(subset=["year", "round", "aggregate_mark", "rank", "college_code", "branch_code", "allotted_category"])
    df["year"] = df["year"].astype(int)
    df["round"] = df["round"].astype(int)

    key_cols = ["year", "round", "college_code", "branch_code", "allotted_category"]
    grouped = (
        df.groupby(key_cols, as_index=False)
        .agg(
            closing_aggregate_mark=("aggregate_mark", "min"),
            closing_general_rank=("rank", "max"),
            median_aggregate_mark=("aggregate_mark", "median"),
            median_general_rank=("rank", "median"),
            allotment_count=("rank", "size"),
            student_community_count=("student_community", "nunique"),
        )
    )
    year_rank_ceiling = df.groupby("year")["rank"].max().rename("year_rank_ceiling")
    grouped = grouped.join(year_rank_ceiling, on="year")
    grouped["closing_rank_percentile"] = grouped["closing_general_rank"] / grouped["year_rank_ceiling"]
    grouped["closing_rank_percentile"] = grouped["closing_rank_percentile"].clip(lower=0.0, upper=1.0)
    grouped["closing_mark_difficulty"] = (grouped["closing_aggregate_mark"] / 200.0).clip(lower=0.0, upper=1.0)
    grouped["selectivity_score"] = ((1.0 - grouped["closing_rank_percentile"]) + grouped["closing_mark_difficulty"]) / 2.0

    recent_keys = grouped[grouped["year"] >= TARGET_YEAR - 3][["round", "college_code", "branch_code", "allotted_category"]].drop_duplicates()
    future = recent_keys.copy()
    future.insert(0, "year", TARGET_YEAR)

    panel = pd.concat([grouped, future], ignore_index=True, sort=False)
    board_features = prepare_board_difficulty_features(board_difficulty)
    panel = panel.merge(board_features, on="year", how="left")
    panel = add_history_features(panel, ["round", "college_code", "branch_code", "allotted_category"], ["closing_aggregate_mark", "closing_general_rank", "allotment_count", "median_aggregate_mark", "median_general_rank", "closing_rank_percentile", "closing_mark_difficulty", "selectivity_score"])
    difficulty_groups = [
        (["college_code"], "difficulty_college_selectivity"),
        (["branch_code"], "difficulty_branch_selectivity"),
        (["allotted_category"], "difficulty_category_selectivity"),
        (["college_code", "branch_code"], "difficulty_college_branch_selectivity"),
        (["round", "branch_code", "allotted_category"], "difficulty_round_branch_category_selectivity"),
    ]
    for group_cols, prefix in difficulty_groups:
        panel = add_prior_group_stats(panel, group_cols, "selectivity_score", prefix)
    feature_defaults = [c for c in panel.columns if c.startswith(("prev_", "roll3_", "trend_"))]
    feature_defaults.extend([c for c in panel.columns if c.startswith("difficulty_")])
    for col in feature_defaults:
        panel[col] = panel[col].fillna(0.0)
    panel["missing_history_count"] = panel["missing_history_count"].fillna(3).astype(float)

    training = panel[panel["year"] < TARGET_YEAR].dropna(subset=["closing_aggregate_mark", "closing_general_rank"]).copy()
    future_inputs = panel[panel["year"] == TARGET_YEAR].copy()
    return training, future_inputs

cutoff_training, cutoff_2026_input = build_cutoff_feature_tables(allotment_raw, board_difficulty_raw)
cutoff_training.to_csv(OUTPUT_DIR / "college_cutoff_training.csv", index=False)
cutoff_2026_input.to_csv(OUTPUT_DIR / "college_cutoff_2026_input.csv", index=False)
print("Cutoff training rows:", len(cutoff_training))
print("Cutoff 2026 input rows:", len(cutoff_2026_input))
cutoff_training.head()


In [ ]:
# Train college cutoff regressor and export 2026 predictions
cutoff_numeric_features = [
    "year", "round", "missing_history_count",
    "hse2_appeared_total", "hse2_pass_percentage", "hse2_fail_percentage", "hse2_competition_index", "hse2_pass_percentage_change", "hse2_appeared_change", "hse2_gender_pass_gap",
    "prev_closing_aggregate_mark", "roll3_median_closing_aggregate_mark", "roll3_std_closing_aggregate_mark", "trend_closing_aggregate_mark",
    "prev_closing_general_rank", "roll3_median_closing_general_rank", "roll3_std_closing_general_rank", "trend_closing_general_rank",
    "prev_closing_rank_percentile", "roll3_median_closing_rank_percentile", "roll3_std_closing_rank_percentile", "trend_closing_rank_percentile",
    "prev_closing_mark_difficulty", "roll3_median_closing_mark_difficulty", "roll3_std_closing_mark_difficulty", "trend_closing_mark_difficulty",
    "prev_selectivity_score", "roll3_median_selectivity_score", "roll3_std_selectivity_score", "trend_selectivity_score",
    "prev_allotment_count", "roll3_median_allotment_count",
    "prev_median_aggregate_mark", "roll3_median_median_aggregate_mark",
    "prev_median_general_rank", "roll3_median_median_general_rank",
    "difficulty_college_selectivity_prev", "difficulty_college_selectivity_roll3_median", "difficulty_college_selectivity_roll3_std", "difficulty_college_selectivity_trend",
    "difficulty_branch_selectivity_prev", "difficulty_branch_selectivity_roll3_median", "difficulty_branch_selectivity_roll3_std", "difficulty_branch_selectivity_trend",
    "difficulty_category_selectivity_prev", "difficulty_category_selectivity_roll3_median", "difficulty_category_selectivity_roll3_std", "difficulty_category_selectivity_trend",
    "difficulty_college_branch_selectivity_prev", "difficulty_college_branch_selectivity_roll3_median", "difficulty_college_branch_selectivity_roll3_std", "difficulty_college_branch_selectivity_trend",
    "difficulty_round_branch_category_selectivity_prev", "difficulty_round_branch_category_selectivity_roll3_median", "difficulty_round_branch_category_selectivity_roll3_std", "difficulty_round_branch_category_selectivity_trend",
]
cutoff_categorical_features = ["college_code", "branch_code", "allotted_category"]

for col in cutoff_numeric_features:
    if col not in cutoff_training.columns:
        cutoff_training[col] = 0.0
        cutoff_2026_input[col] = 0.0
    cutoff_training[col] = pd.to_numeric(cutoff_training[col], errors="coerce").fillna(0.0)
    cutoff_2026_input[col] = pd.to_numeric(cutoff_2026_input[col], errors="coerce").fillna(0.0)

cutoff_preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), cutoff_numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cutoff_categorical_features),
    ]
)

cutoff_train_df = cutoff_training[cutoff_training["year"] < VALIDATION_YEAR].copy()
cutoff_val_df = cutoff_training[cutoff_training["year"] == VALIDATION_YEAR].copy()

X_cutoff_train = to_dense_float32(cutoff_preprocessor.fit_transform(cutoff_train_df[cutoff_numeric_features + cutoff_categorical_features]))
X_cutoff_val = to_dense_float32(cutoff_preprocessor.transform(cutoff_val_df[cutoff_numeric_features + cutoff_categorical_features]))
X_cutoff_2026 = to_dense_float32(cutoff_preprocessor.transform(cutoff_2026_input[cutoff_numeric_features + cutoff_categorical_features]))

y_cutoff_train_raw = np.column_stack([
    cutoff_train_df["closing_aggregate_mark"].astype(float).to_numpy(),
    np.log1p(cutoff_train_df["closing_general_rank"].astype(float).to_numpy()),
])
y_cutoff_val_raw = np.column_stack([
    cutoff_val_df["closing_aggregate_mark"].astype(float).to_numpy(),
    np.log1p(cutoff_val_df["closing_general_rank"].astype(float).to_numpy()),
])
target_scaler = StandardScaler()
y_cutoff_train = target_scaler.fit_transform(y_cutoff_train_raw).astype("float32")
y_cutoff_val = target_scaler.transform(y_cutoff_val_raw).astype("float32")

cutoff_model = make_dense_regressor(X_cutoff_train.shape[1], 2)
cutoff_param_count = cutoff_model.count_params()
print(f"Cutoff model trainable parameters: {cutoff_param_count:,}")
if cutoff_param_count < 1_000_000:
    raise RuntimeError("Cutoff model is below 1M parameters. Restart the Colab runtime and rerun all cells from the updated notebook.")
cutoff_model.summary()
cutoff_callbacks = [tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=25, min_lr=1e-6)]
cutoff_model.fit(
    X_cutoff_train,
    y_cutoff_train,
    validation_data=(X_cutoff_val, y_cutoff_val),
    epochs=500,
    batch_size=512,
    callbacks=cutoff_callbacks,
    verbose=1,
)

cutoff_val_pred_scaled = cutoff_model.predict(X_cutoff_val, verbose=0)
cutoff_val_pred = target_scaler.inverse_transform(cutoff_val_pred_scaled)
cutoff_val_pred_mark = cutoff_val_pred[:, 0]
cutoff_val_pred_rank = np.expm1(cutoff_val_pred[:, 1])

cutoff_2026_pred_scaled = cutoff_model.predict(X_cutoff_2026, verbose=0)
cutoff_2026_pred = target_scaler.inverse_transform(cutoff_2026_pred_scaled)
cutoff_predictions = cutoff_2026_input[["year", "round", "college_code", "branch_code", "allotted_category", "missing_history_count"]].copy()
cutoff_predictions["predicted_closing_aggregate_mark"] = np.clip(cutoff_2026_pred[:, 0], 0, 200)
cutoff_predictions["predicted_closing_general_rank"] = np.maximum(1, np.expm1(cutoff_2026_pred[:, 1])).round().astype(int)
cutoff_predictions.to_csv(OUTPUT_DIR / "college_cutoff_2026_predictions.csv", index=False)

cutoff_model.save(OUTPUT_DIR / "college_cutoff_model.keras")
joblib.dump({"preprocessor": cutoff_preprocessor, "target_scaler": target_scaler, "features": cutoff_numeric_features + cutoff_categorical_features}, OUTPUT_DIR / "cutoff_feature_pipeline.joblib")

cutoff_metrics = {
    "closing_mark_mae": float(mean_absolute_error(cutoff_val_df["closing_aggregate_mark"], cutoff_val_pred_mark)),
    "closing_rank_mae": float(mean_absolute_error(cutoff_val_df["closing_general_rank"], cutoff_val_pred_rank)),
    "model_params": int(cutoff_param_count),
    "epochs_requested": 500,
    "train_rows": int(len(cutoff_train_df)),
    "validation_rows": int(len(cutoff_val_df)),
    "prediction_rows_2026": int(len(cutoff_predictions)),
}
cutoff_metrics


In [ ]:
# Save metrics and download artifacts
metrics = {
    "target_year": TARGET_YEAR,
    "validation_year": VALIDATION_YEAR,
    "rank_band_model": rank_metrics,
    "college_cutoff_model": cutoff_metrics,
    "source_rows": {
        "grl": int(len(grl_raw)),
        "allotment": int(len(allotment_raw)),
    },
}

metrics_path = OUTPUT_DIR / "metrics.json"
metrics_path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")
print(json.dumps(metrics, indent=2))

artifact_paths = [
    OUTPUT_DIR / "rank_band_training.csv",
    OUTPUT_DIR / "rank_band_2026_input.csv",
    OUTPUT_DIR / "rank_band_2026_predictions.csv",
    OUTPUT_DIR / "general_rank_band_model.keras",
    OUTPUT_DIR / "community_rank_band_model.keras",
    OUTPUT_DIR / "rank_feature_pipeline.joblib",
    OUTPUT_DIR / "college_cutoff_training.csv",
    OUTPUT_DIR / "college_cutoff_2026_input.csv",
    OUTPUT_DIR / "college_cutoff_2026_predictions.csv",
    OUTPUT_DIR / "college_cutoff_model.keras",
    OUTPUT_DIR / "cutoff_feature_pipeline.joblib",
    metrics_path,
]

for path in artifact_paths:
    print(path, path.stat().st_size if path.exists() else "missing")

# Uncomment if you want Colab to download every artifact to your machine.
# for path in artifact_paths:
#     files.download(str(path))
